# PathSnap: KNN on the existing preprocessed dataset

This notebook trains and evaluates **K-Nearest Neighbors (KNN)** using the already-created `shared_data/train_candidates.npz` and `shared_data/student_test_candidates.npz` files.

**It does not rerun preprocessing.** Each saved row is already one GPS-observation/road-candidate pair with six features. This notebook scales those features, evaluates KNN with five-fold cross-validation grouped by trip, fits one final KNN model on all training rows, evaluates it on the supplied student-test rows, and saves separate KNN outputs.

KNN is distance-based, so feature scaling matters: otherwise metre distances and radians would have incomparable numeric ranges. This full-data run can take a while on a laptop because KNN searches through many stored training examples for every query.


In [1]:
from pathlib import Path
import json
import time
import numpy as np
from joblib import dump
from sklearn.metrics import f1_score, log_loss
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Find this GitHub bundle whether Jupyter starts in the bundle or repo root.
start = Path.cwd().resolve()
roots = [start, *start.parents]
bundle_candidates = [start / "KNN"]
bundle_candidates.extend(p for p in roots if p.name == "KNN")
BUNDLE = next(
    (p for p in bundle_candidates if (p / "02_knn_same_preprocessed_data.ipynb").is_file()),
    None,
)
if BUNDLE is None:
    raise FileNotFoundError(
        "Could not locate KNN. Open this notebook from its folder or the repository root."
    )

# Preferred location is this folder's shared_data/. Fallbacks support a shared
# dataset stored alongside the bundle or in the teammate's PathSnap workspace.
data_candidates = [
    BUNDLE / "shared_data",
    BUNDLE.parent / "shared_data",
    BUNDLE.parent / "pathsnap" / "shared_data",
    BUNDLE.parent.parent / "pathsnap" / "shared_data",
]
required_files = [
    "train_candidates.npz", "train_metadata.json",
    "student_test_candidates.npz", "student_test_metadata.json",
]
DATA = next(
    (p for p in data_candidates if all((p / name).is_file() for name in required_files)),
    None,
)
if DATA is None:
    raise FileNotFoundError(
        "Preprocessed files not found. Copy the four train/student-test NPZ and metadata files into this folder's shared_data/ directory. See README.md."
    )
RESULTS = BUNDLE / "results"
ARTIFACTS = BUNDLE / "artifacts"
RESULTS.mkdir(exist_ok=True)
ARTIFACTS.mkdir(exist_ok=True)

train_file = np.load(DATA / "train_candidates.npz", allow_pickle=False)
test_file = np.load(DATA / "student_test_candidates.npz", allow_pickle=False)
train_meta = json.loads((DATA / "train_metadata.json").read_text())
test_meta = json.loads((DATA / "student_test_metadata.json").read_text())

X = train_file["X"].astype(np.float32, copy=False)
y = train_file["y"]
groups = train_file["trip_id"]
train_obs = train_file["gps_observation_id"]
X_test = test_file["X"].astype(np.float32, copy=False)
y_test = test_file["y"]
test_obs = test_file["gps_observation_id"]
feature_names = train_file["feature_names"].astype(str).tolist()

assert X.shape == (1_145_304, 6), f"Unexpected train data shape: {X.shape}"
assert X_test.shape == (450_648, 6), f"Unexpected student-test data shape: {X_test.shape}"
assert feature_names == train_meta["feature_names"]

print("KNN bundle:", BUNDLE)
print("Preprocessed data:", DATA)
print("Train X:", X.shape, "Student-test X:", X_test.shape)
print("Features:", feature_names)
print("Train trips:", len(np.unique(groups)))
print("Train sampled GPS points:", train_meta["sampled_or_total_gps_points"])
print("Student-test GPS points:", test_meta["sampled_or_total_gps_points"])


Project: KNN (original run)
Train X: (1145304, 6) Student-test X: (450648, 6)
Features: ['cross_track_m', 'heading_error_rad', 'along_fraction', 'segment_length_m', 'gps_speed_mps', 'time_gap_s']
Train trips: 80
Train sampled GPS points: 49664
Student-test GPS points: 19160


## What KNN will learn

KNN does not build a compact set of decision rules like a tree. It keeps the labelled training examples. For a new candidate row, it finds the **15 closest training rows** in the six-feature space and uses their labels to estimate the candidate's positive probability. We use distance weighting, so closer neighbours count more.

The pipeline first applies `StandardScaler`, which centres each feature and scales it to unit variance. During cross-validation the scaler is fitted only on that fold's training trips, preventing validation data from influencing scaling. `KDTree` is used to find neighbours; it is a practical index for this low-dimensional feature set, though searches over this many rows can still be computationally expensive.

The observation-level top-1 score groups each point's 24 candidates by `gps_observation_id` and checks whether KNN ranked the positive candidate highest. Candidate-level macro-F1 uses a probability threshold of 0.5 and answers a different question.


In [2]:
# Match the KNN configuration used in the earlier model screen.
KNN_PARAMS = {
    "n_neighbors": 15,
    "weights": "distance",
    "algorithm": "kd_tree",
    "leaf_size": 40,
    "n_jobs": 4,
}
N_SPLITS = 5
BATCH_SIZE = 4_000


def make_model():
    return make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(**KNN_PARAMS),
    )


def predict_probabilities_batched(model, features, batch_size=BATCH_SIZE):
    """Predict positive-class probabilities in chunks to bound temporary memory."""
    predictions = []
    for start in range(0, len(features), batch_size):
        stop = min(start + batch_size, len(features))
        predictions.append(model.predict_proba(features[start:stop])[:, 1])
        if stop == len(features) or stop % 40_000 == 0:
            print(f"Scored {stop:,} / {len(features):,} candidate rows", flush=True)
    return np.concatenate(predictions)


def observation_top1(y_true, probabilities, observation_ids):
    """Top-scoring candidate per observation; stable first-row tie handling."""
    _, inverse = np.unique(observation_ids, return_inverse=True)
    n_observations = int(inverse.max()) + 1
    best_probability = np.full(n_observations, -np.inf)
    np.maximum.at(best_probability, inverse, probabilities)
    is_winner = probabilities == best_probability[inverse]
    winning_rows = np.flatnonzero(is_winner)
    first_winner = np.full(n_observations, len(y_true), dtype=np.int64)
    np.minimum.at(first_winner, inverse[winning_rows], winning_rows)
    return float(np.mean(y_true[first_winner] == 1)), n_observations


def make_cv_pipeline_metrics(y_true, probabilities):
    return float(f1_score(y_true, probabilities >= 0.5, average="macro"))

print("KNN parameters:", KNN_PARAMS)
print("CV folds:", N_SPLITS, "query batch size:", BATCH_SIZE)


KNN parameters: {'n_neighbors': 15, 'weights': 'distance', 'algorithm': 'kd_tree', 'leaf_size': 40, 'n_jobs': 4}
CV folds: 5 query batch size: 4000


## 1. Five-fold cross-validation, grouped by trip

We split using `trip_id`, not random candidate rows. This keeps every candidate from one trip on the same side of a fold. For each fold, the scaler and KNN are fitted on the training trips, then probabilities are computed for the held-out trips in batches.

This is the slowest cell. Let it finish; each fold prints progress. It writes its real measurements to `results/knn_cv.json`.


In [3]:
cv = GroupKFold(n_splits=N_SPLITS)
fold_results = []
cv_started = time.perf_counter()

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y, groups), start=1):
    print(f"\nFold {fold}/{N_SPLITS}: fit on {len(train_idx):,} rows; validate on {len(valid_idx):,} rows", flush=True)
    model = make_model()
    fit_started = time.perf_counter()
    model.fit(X[train_idx], y[train_idx])
    fit_seconds = time.perf_counter() - fit_started

    predict_started = time.perf_counter()
    valid_probabilities = predict_probabilities_batched(model, X[valid_idx])
    predict_seconds = time.perf_counter() - predict_started
    top1, n_observations = observation_top1(y[valid_idx], valid_probabilities, train_obs[valid_idx])
    macro_f1 = make_cv_pipeline_metrics(y[valid_idx], valid_probabilities)
    result = {
        "fold": fold,
        "held_out_trips": sorted(np.unique(groups[valid_idx]).astype(str).tolist()),
        "candidate_rows": int(len(valid_idx)),
        "observations": int(n_observations),
        "observation_top1": top1,
        "candidate_macro_f1_at_0.5": macro_f1,
        "fit_seconds": fit_seconds,
        "prediction_seconds": predict_seconds,
    }
    fold_results.append(result)
    print(result, flush=True)

cv_seconds = time.perf_counter() - cv_started
cv_summary = {
    "model": "KNeighborsClassifier",
    "parameters": KNN_PARAMS,
    "features": feature_names,
    "train_rows": int(len(y)),
    "train_trips": int(len(np.unique(groups))),
    "cv_method": "5-fold GroupKFold by trip_id",
    "observation_top1_mean": float(np.mean([r["observation_top1"] for r in fold_results])),
    "observation_top1_std": float(np.std([r["observation_top1"] for r in fold_results])),
    "candidate_macro_f1_mean": float(np.mean([r["candidate_macro_f1_at_0.5"] for r in fold_results])),
    "folds": fold_results,
    "total_cv_seconds": cv_seconds,
}
(RESULTS / "knn_cv.json").write_text(json.dumps(cv_summary, indent=2))
print("\nCV summary:")
print(json.dumps({k: v for k, v in cv_summary.items() if k != "folds"}, indent=2))



Fold 1/5: fit on 917,400 rows; validate on 227,904 rows
Scored 40,000 / 227,904 candidate rows
Scored 80,000 / 227,904 candidate rows
Scored 120,000 / 227,904 candidate rows
Scored 160,000 / 227,904 candidate rows
Scored 200,000 / 227,904 candidate rows
Scored 227,904 / 227,904 candidate rows
{'fold': 1, 'held_out_trips': ['00000006', '00000012', '00000019', '00000021', '00000031', '00000033', '00000042', '00000044', '00000053', '00000063', '00000074', '00000079', '00000080', '00000082', '00000083', '00000084'], 'candidate_rows': 227904, 'observations': 9496, 'observation_top1': 0.7392586352148273, 'candidate_macro_f1_at_0.5': 0.8369438355918025, 'fit_seconds': 3.7571444530003646, 'prediction_seconds': 29.565941946002567}

Fold 2/5: fit on 915,144 rows; validate on 230,160 rows
Scored 40,000 / 230,160 candidate rows
Scored 80,000 / 230,160 candidate rows
Scored 120,000 / 230,160 candidate rows
Scored 160,000 / 230,160 candidate rows
Scored 200,000 / 230,160 candidate rows
Scored 230,1

## 2. Fit the final KNN and evaluate the student-test partition

After cross-validation, fit a fresh scaler + KNN pipeline on **all** preprocessed training candidate rows. Then score all student-test candidates in batches. The student-test data is for development evaluation, not the reserved evaluator partition.

The model and metrics are saved separately from the Random Forest files as `artifacts/knn.joblib` and `results/knn_results.json`.


In [4]:
final_model = make_model()
fit_started = time.perf_counter()
final_model.fit(X, y)
final_fit_seconds = time.perf_counter() - fit_started

dump(final_model, ARTIFACTS / "knn.joblib")
print(f"Saved KNN model: {ARTIFACTS / 'knn.joblib'}")
print(f"Final fit time: {final_fit_seconds:.2f} seconds")

infer_started = time.perf_counter()
test_probabilities = predict_probabilities_batched(final_model, X_test)
inference_seconds = time.perf_counter() - infer_started
student_top1, covered_observations = observation_top1(y_test, test_probabilities, test_obs)
student_macro_f1 = float(f1_score(y_test, test_probabilities >= 0.5, average="macro"))
student_log_loss = float(log_loss(y_test, np.c_[1 - test_probabilities, test_probabilities], labels=[0, 1]))

knn_results = {
    "model": "KNeighborsClassifier",
    "parameters": KNN_PARAMS,
    "features": feature_names,
    "train_candidate_rows": int(len(y)),
    "train_gps_points": int(train_meta["sampled_or_total_gps_points"]),
    "train_candidate_miss_rate": float(train_meta["candidate_miss_rate"]),
    "student_test_candidate_rows": int(len(y_test)),
    "student_test_gps_points": int(test_meta["sampled_or_total_gps_points"]),
    "student_test_covered_points": int(covered_observations),
    "student_test_candidate_miss_rate": float(test_meta["candidate_miss_rate"]),
    "grouped_5fold_cv_observation_top1_mean": cv_summary["observation_top1_mean"],
    "grouped_5fold_cv_observation_top1_std": cv_summary["observation_top1_std"],
    "grouped_5fold_cv_candidate_macro_f1_mean": cv_summary["candidate_macro_f1_mean"],
    "student_test_observation_top1": student_top1,
    "student_test_candidate_macro_f1_at_0.5": student_macro_f1,
    "student_test_candidate_log_loss": student_log_loss,
    "final_fit_seconds": final_fit_seconds,
    "student_test_inference_seconds": inference_seconds,
    "artifact": "artifacts/knn.joblib",
    "evaluation_note": "Top-1 is conditional on a positive reference candidate being generated. Labels are pointwise geometric pseudo-labels; these are candidate-ranking results, not sequence-level route accuracy.",
}
(RESULTS / "knn_results.json").write_text(json.dumps(knn_results, indent=2))
print(json.dumps(knn_results, indent=2))


Saved KNN model: /media/andy/7CF6C4F2F6C4AE28/Users/Andy/Desktop/COLLEGE/ML/hackathon/pathsnap/artifacts/knn.joblib
Final fit time: 2.41 seconds
Scored 40,000 / 450,648 candidate rows
Scored 80,000 / 450,648 candidate rows
Scored 120,000 / 450,648 candidate rows
Scored 160,000 / 450,648 candidate rows
Scored 200,000 / 450,648 candidate rows
Scored 240,000 / 450,648 candidate rows
Scored 280,000 / 450,648 candidate rows
Scored 320,000 / 450,648 candidate rows
Scored 360,000 / 450,648 candidate rows
Scored 400,000 / 450,648 candidate rows
Scored 440,000 / 450,648 candidate rows
Scored 450,648 / 450,648 candidate rows
{
  "model": "KNeighborsClassifier",
  "parameters": {
    "n_neighbors": 15,
    "weights": "distance",
    "algorithm": "kd_tree",
    "leaf_size": 40,
    "n_jobs": 4
  },
  "features": [
    "cross_track_m",
    "heading_error_rad",
    "along_fraction",
    "segment_length_m",
    "gps_speed_mps",
    "time_gap_s"
  ],
  "train_candidate_rows": 1145304,
  "train_gps_poi

## 3. How to read the result

- **Observation top-1**: among covered GPS observations, the fraction for which KNN ranked the positive road candidate first.
- **Candidate macro-F1**: candidate-row classification quality at probability 0.5, with positive and negative classes weighted equally.
- **Candidate miss rate**: fraction of GPS observations whose reference road was absent from the candidate list. The classifier cannot fix these misses.
- **Log loss**: evaluates the quality/confidence of candidate probabilities; lower is better.
- **Fit and inference seconds**: useful for comparing KNN's computational cost with the Random Forest run.

The results are not comparable to the earlier small model screen unless that screen is rerun with the exact same processed train/test files, sampling, and evaluation procedure. This notebook uses the full saved feature tables and will record its own exact dataset shapes/settings in the JSON result files.

KNN is memory-light compared with a large neural network but can be prediction-heavy: it stores the training rows and searches them for each new candidate. It also does not use road-network continuity. HMM/Viterbi is still needed for a coherent route sequence.
